In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

In [5]:
df = pd.read_excel("customer_intent_classification_dataset.xlsx")
df.head()

,Row_ID,Customer_Utterance,Intent_Label,Intent_Score,Intent_Description,Sentiment,Channel,Product_Category,Industry,Country,Customer_Age_Group,Device,Prior_Interactions,Session_Duration_Min,CTA_Clicked,Follow_Up_Status
0,1,I'd like to understand the full scope before d...,Neutral,3,Open to purchase but undecided,Neutral,Email,SaaS Tool,Media,Canada,55+,Tablet,1,14.7,NaN,Pending
1,2,Can you confirm the price? I'm ready to place ...,Strongly Interested,5,"High purchase intent, ready to buy",Very Positive,Mobile App,SaaS Tool,Travel,Australia,18-24,Mobile,5,12.4,Subscribe,Yes - Follow-up Scheduled
2,3,I've compared all options and I want this one....,Strongly Interested,5,"High purchase intent, ready to buy",Very Positive,Mobile App,E-Commerce Product,Retail,Canada,18-24,Desktop,4,13.9,Subscribe,Yes - Closed
3,4,This seems like a good fit. What are the next ...,Interested,4,"Considering purchase, has shown clear intent",Slightly Positive,Email,Travel Package,Media,India,45-54,Mobile,4,19.2,Get a Quote,Pending
4,5,I've compared all options and I want this one....,Strongly Interested,5,"High purchase intent, ready to buy",Very Positive,Email,SaaS Tool,Technology,UAE,45-54,Desktop,6,20.6,Subscribe,Yes - Follow-up Scheduled


In [7]:
df = df[['Customer_Utterance',
         'Session_Duration_Min',
         'Prior_Interactions',
         'Follow_Up_Status',
         'Intent_Label']]

df.dropna(inplace=True)
df.head()

,Customer_Utterance,Session_Duration_Min,Prior_Interactions,Follow_Up_Status,Intent_Label
0,I'd like to understand the full scope before d...,14.7,1,Pending,Neutral
1,Can you confirm the price? I'm ready to place ...,12.4,5,Yes - Follow-up Scheduled,Strongly Interested
2,I've compared all options and I want this one....,13.9,4,Yes - Closed,Strongly Interested
3,This seems like a good fit. What are the next ...,19.2,4,Pending,Interested
4,I've compared all options and I want this one....,20.6,6,Yes - Follow-up Scheduled,Strongly Interested


In [8]:
le_follow = LabelEncoder()
df['Follow_Up_Status'] = le_follow.fit_transform(df['Follow_Up_Status'])

le_target = LabelEncoder()
df['Intent_Label'] = le_target.fit_transform(df['Intent_Label'])

In [9]:
tfidf = TfidfVectorizer(max_features=1000)

X_text = tfidf.fit_transform(df['Customer_Utterance']).toarray()

In [10]:
X_other = df[['Session_Duration_Min', 'Prior_Interactions', 'Follow_Up_Status']].values

X = np.hstack((X_text, X_other))
y = df['Intent_Label']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    eval_metric='mlogloss'
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [14]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9807692307692307

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.93      0.97        15
           1       0.96      1.00      0.98        22
           2       1.00      1.00      1.00        24
           3       0.95      1.00      0.98        21
           4       1.00      0.95      0.98        22

    accuracy                           0.98       104
   macro avg       0.98      0.98      0.98       104
weighted avg       0.98      0.98      0.98       104



In [19]:
def predict_intent_full():
    # 1. Take user inputs
    utterance = input("Enter customer utterance: ")

    session_duration = float(input("Enter session duration (in minutes): "))

    prior_interactions = int(input("Enter number of prior interactions: "))

    # Show available follow-up options
    print("\nFollow-Up Status Options:")
    for i, label in enumerate(le_follow.classes_):
        print(f"{i}: {label}")

    follow_up_choice = int(input("Select follow-up status (enter number): "))

    follow_up = follow_up_choice

    # 2. Transform text
    text_vec = tfidf.transform([utterance]).toarray()

    # 3. Combine features
    other_features = np.array([[session_duration, prior_interactions, follow_up]])
    final_input = np.hstack((text_vec, other_features))

    # 4. Predict
    pred = model.predict(final_input)

    predicted_label = le_target.inverse_transform(pred)[0]

    print("\n🔮 Predicted Intent:", predicted_label)

In [20]:
predict_intent_full()

Enter customer utterance: OMG I WNAT TO BUY THIS
Enter session duration (in minutes): 23
Enter number of prior interactions: 8

Follow-Up Status Options:
0: No
1: Pending
2: Unsubscribed
3: Yes - Closed
4: Yes - Follow-up Scheduled
Select follow-up status (enter number): 4

🔮 Predicted Intent: Strongly Interested
